# Notebook 12 — Structured Outputs, Tool Calls, and Response Contracts

Modern serving layers do more than emit text. They enforce JSON contracts, normalize tool calls, and decide whether malformed output should be repaired, rejected, or retried. This notebook keeps the whole stack notebook-first and open-source by building the contract layers directly in Python.

## What you will build

- a structured JSON contract using `pydantic`
- parser layers that recover JSON from fenced or mixed prose output
- a normalized tool-call contract plus an OpenAI-compatible adapter
- contract-pass simulations across several serving policies
- a failure-mode and repair report you can reuse in later runtime notebooks

## Why this belongs in the serving layer

Earlier notebooks treated the runtime as a scheduler and cache manager. By 2026, the runtime also owns output shape: schema-constrained JSON, tool invocation payloads, and provider-compatible response envelopes. That means correctness is no longer “did the model answer?” It is also “did the service preserve the contract?”

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add helpers and an artifact directory

We will use plain Python plus `pydantic`. The point is to expose the parser and validator layers instead of hiding them behind a managed API.

In [ ]:
import copy
import re
from pydantic import BaseModel, Field, ValidationError
from typing import Any, Literal

random.seed(12)
ARTIFACT_DIR = ARTIFACT_ROOT / "12_structured_outputs_tool_calls_and_contracts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def model_schema(model_cls):
    if hasattr(model_cls, "model_json_schema"):
        return model_cls.model_json_schema()
    return model_cls.schema()

def model_validate(model_cls, payload):
    if hasattr(model_cls, "model_validate"):
        return model_cls.model_validate(payload)
    return model_cls.parse_obj(payload)

def clipped(value, low=0.0, high=1.0):
    return max(low, min(high, value))

def weighted_pick(weight_map, rng):
    roll = rng.random()
    cursor = 0.0
    items = list(weight_map.items())
    for name, weight in items:
        cursor += weight
        if roll <= cursor:
            return name
    return items[-1][0]

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define a structured JSON contract

This contract is intentionally small. The interesting part is not the task itself. It is the guarantee that every valid response contains the same fields, types, and confidence range.

In [ ]:
class StructuredAnswer(BaseModel):
    request_id: str
    answer_type: Literal["summary", "decision", "warning"]
    summary: str = Field(min_length=12, max_length=280)
    confidence: float = Field(ge=0.0, le=1.0)
    citations: list[str] = Field(min_length=1)
    needs_human_review: bool

structured_schema = model_schema(StructuredAnswer)
schema_rows = []
for field_name, property_spec in structured_schema["properties"].items():
    type_display = json.dumps(property_spec.get("type", property_spec.get("anyOf", [])))
    schema_rows.append({
        "field": field_name,
        "required": field_name in structured_schema.get("required", []),
        "type": type_display,
        "description": property_spec.get("title", ""),
    })

pd.DataFrame(schema_rows)

## Step 3: Create notebook-local response cases

We want both valid and invalid examples: clean JSON, fenced JSON, prose wrapped around JSON, missing fields, wrong enums, and malformed payloads.

In [ ]:
structured_cases = [
    {
        "case": "clean_json",
        "raw": json.dumps({
            "request_id": "req-001",
            "answer_type": "summary",
            "summary": "Prefix caching reduces repeated prefill work for shared scaffolds.",
            "confidence": 0.91,
            "citations": ["notebook-11", "runtime-cache-guide"],
            "needs_human_review": False,
        }),
    },
    {
        "case": "fenced_json",
        "raw": "```json\n" + json.dumps({
            "request_id": "req-002",
            "answer_type": "decision",
            "summary": "Use a schema gate when downstream code expects exact keys.",
            "confidence": 0.83,
            "citations": ["contract-playbook"],
            "needs_human_review": False,
        }, indent=2) + "\n```",
    },
    {
        "case": "prose_then_json",
        "raw": "Draft answer follows.\n" + json.dumps({
            "request_id": "req-003",
            "answer_type": "warning",
            "summary": "Free-form completions can hide valid JSON inside extra prose.",
            "confidence": 0.72,
            "citations": ["serving-notes"],
            "needs_human_review": True,
        }),
    },
    {
        "case": "string_citations",
        "raw": json.dumps({
            "request_id": "req-004",
            "answer_type": "summary",
            "summary": "The model used a string where the contract expected a list.",
            "confidence": 0.67,
            "citations": "ops-runbook",
            "needs_human_review": False,
        }),
    },
    {
        "case": "missing_review_flag",
        "raw": json.dumps({
            "request_id": "req-005",
            "answer_type": "decision",
            "summary": "Required booleans matter because callers should not guess defaults.",
            "confidence": 0.64,
            "citations": ["validation-guide"],
        }),
    },
    {
        "case": "broken_json",
        "raw": "{\"request_id\": \"req-006\", \"answer_type\": \"summary\", \"summary\": \"The parser never receives a closed JSON object.\", \"confidence\": 0.55, \"citations\": [\"ops-runbook\"], \"needs_human_review\": false",
    },
    {
        "case": "wrong_enum",
        "raw": json.dumps({
            "request_id": "req-007",
            "answer_type": "analysis",
            "summary": "Unknown enum values should fail before application code sees them.",
            "confidence": 0.79,
            "citations": ["response-contracts"],
            "needs_human_review": False,
        }),
    },
]

pd.DataFrame([{
    "case": row["case"],
    "characters": len(row["raw"]),
    "preview": row["raw"][:72],
} for row in structured_cases])

## Step 4: Build the parser layer

The parser layer is where brittle integrations usually fail. We will normalize fences, extract the first balanced JSON object, and only then attempt a strict parse.

In [ ]:
def strip_code_fences(text):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```[a-zA-Z0-9_-]*\n?", "", cleaned)
        cleaned = re.sub(r"\n?```$", "", cleaned)
    return cleaned.strip()

def extract_first_json_object(text):
    start = text.find("{")
    if start < 0:
        return ""
    depth = 0
    in_string = False
    escape = False
    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return text[start:index + 1]
    return text[start:]

def parse_json_candidate(raw_text):
    cleaned = strip_code_fences(raw_text)
    candidate = extract_first_json_object(cleaned)
    if not candidate:
        return {"candidate": "", "payload": None, "error": "no_json_object_found"}
    try:
        return {"candidate": candidate, "payload": json.loads(candidate), "error": None}
    except json.JSONDecodeError as exc:
        return {"candidate": candidate, "payload": None, "error": f"{exc.msg} at char {exc.pos}"}

parse_json_candidate(structured_cases[1]["raw"])

## Step 5: Validate the structured contract

Parsing JSON is necessary but insufficient. Contract validation catches missing fields, wrong enums, and shape mismatches before they propagate into routing or tool layers.

In [ ]:
def validate_structured_payload(payload):
    try:
        validated = model_validate(StructuredAnswer, payload)
        dumped = validated.model_dump() if hasattr(validated, "model_dump") else validated.dict()
        return {"ok": True, "normalized": dumped, "errors": []}
    except ValidationError as exc:
        errors = []
        for issue in exc.errors():
            field_path = ".".join(str(part) for part in issue["loc"])
            errors.append(field_path + ": " + issue["msg"])
        return {"ok": False, "normalized": payload, "errors": errors}

def classify_structured_failure(parsed, validated):
    if parsed["error"] is not None:
        return "json_parse_error"
    if validated["ok"]:
        return "ok"
    joined = " | ".join(validated["errors"])
    if "needs_human_review" in joined:
        return "missing_required_field"
    if "citations" in joined:
        return "citations_shape_error"
    if "answer_type" in joined:
        return "enum_mismatch"
    if "confidence" in joined:
        return "type_mismatch"
    return "schema_validation_error"

def run_structured_pipeline(case):
    parsed = parse_json_candidate(case["raw"])
    validated = validate_structured_payload(parsed["payload"]) if parsed["payload"] is not None else {"ok": False, "normalized": None, "errors": []}
    normalized = validated["normalized"] if validated["ok"] else None
    return {
        "case": case["case"],
        "parse_ok": parsed["error"] is None,
        "validation_ok": validated["ok"],
        "failure_mode": classify_structured_failure(parsed, validated),
        "error_count": len(validated["errors"]),
        "top_error": validated["errors"][0] if validated["errors"] else "",
        "normalized_summary": "" if normalized is None else normalized["summary"][:80],
    }

## Step 6: Run the structured pipeline

Notice the separation of concerns: the parser decides whether JSON exists; the validator decides whether the JSON satisfies the schema.

In [ ]:
structured_results = pd.DataFrame([run_structured_pipeline(case) for case in structured_cases])
structured_results

## Step 7: Define a normalized tool-call contract

Tool calling is easiest to reason about when you normalize everything into one internal shape. Providers may disagree on nesting and argument encoding, but your runtime does not have to.

In [ ]:
class NormalizedToolCall(BaseModel):
    id: str
    name: Literal["search_docs", "lookup_metric", "fetch_runbook"]
    arguments: dict[str, Any]

class NormalizedToolTurn(BaseModel):
    role: Literal["assistant"] = "assistant"
    content: Optional[str] = None
    tool_calls: list[NormalizedToolCall] = Field(default_factory=list)
    finish_reason: Literal["tool_calls", "stop", "length"] = "tool_calls"

def parse_argument_value(value):
    if isinstance(value, dict):
        return value, None
    if isinstance(value, str):
        try:
            return json.loads(value), None
        except json.JSONDecodeError as exc:
            return None, f"{exc.msg} at char {exc.pos}"
    return None, "arguments must be a dict or JSON string"

def normalize_tool_response(payload):
    if "choices" in payload:
        choice = payload["choices"][0]
        message = choice.get("message", {})
        finish_reason = choice.get("finish_reason", "tool_calls" if message.get("tool_calls") else "stop")
    else:
        message = payload
        finish_reason = payload.get("finish_reason", "tool_calls" if payload.get("tool_calls") else "stop")
    normalized = {"role": message.get("role", "assistant"), "content": message.get("content"), "tool_calls": [], "finish_reason": finish_reason}
    argument_errors = []
    for index, raw_call in enumerate(message.get("tool_calls", [])):
        if "function" in raw_call:
            function_block = raw_call.get("function", {})
            name = function_block.get("name")
            raw_args = function_block.get("arguments", {})
        else:
            name = raw_call.get("name")
            raw_args = raw_call.get("arguments", {})
        parsed_args, error = parse_argument_value(raw_args)
        if error is not None:
            argument_errors.append(f"tool_calls[{index}]: {error}")
        normalized["tool_calls"].append({
            "id": raw_call.get("id", f"call_{index}"),
            "name": name,
            "arguments": parsed_args if parsed_args is not None else raw_args,
        })
    return normalized, argument_errors

def validate_tool_turn(payload):
    normalized, argument_errors = normalize_tool_response(payload)
    if argument_errors:
        return {"ok": False, "normalized": normalized, "errors": argument_errors}
    try:
        validated = model_validate(NormalizedToolTurn, normalized)
        dumped = validated.model_dump() if hasattr(validated, "model_dump") else validated.dict()
        return {"ok": True, "normalized": dumped, "errors": []}
    except ValidationError as exc:
        errors = []
        for issue in exc.errors():
            field_path = ".".join(str(part) for part in issue["loc"])
            errors.append(field_path + ": " + issue["msg"])
        return {"ok": False, "normalized": normalized, "errors": errors}

## Step 8: Evaluate tool-call examples

We will validate both OpenAI-style nested payloads and a flatter normalized form. The goal is to prove that one internal contract can absorb multiple external response shapes.

In [ ]:
tool_cases = [
    {
        "case": "openai_good",
        "payload": {
            "choices": [
                {
                    "message": {
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [
                            {
                                "id": "call_1",
                                "type": "function",
                                "function": {
                                    "name": "search_docs",
                                    "arguments": json.dumps({"query": "prefix caching", "top_k": 3}),
                                },
                            }
                        ],
                    },
                    "finish_reason": "tool_calls",
                }
            ]
        },
    },
    {
        "case": "normalized_dict_args",
        "payload": {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {"id": "call_2", "name": "lookup_metric", "arguments": {"metric": "p95_latency_ms", "window": "15m"}}
            ],
            "finish_reason": "tool_calls",
        },
    },
    {
        "case": "missing_finish_reason",
        "payload": {
            "choices": [
                {
                    "message": {
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [
                            {
                                "id": "call_3",
                                "type": "function",
                                "function": {
                                    "name": "fetch_runbook",
                                    "arguments": json.dumps({"slug": "contract-validation"}),
                                },
                            }
                        ],
                    }
                }
            ]
        },
    },
    {
        "case": "single_quote_arguments",
        "payload": {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {"id": "call_4", "name": "lookup_metric", "arguments": "{\'metric\': \'queue_ms\', \'window\': \'5m\'}"}
            ],
            "finish_reason": "tool_calls",
        },
    },
    {
        "case": "unknown_tool_name",
        "payload": {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {"id": "call_5", "name": "open_browser", "arguments": {"url": "https://example.com"}}
            ],
            "finish_reason": "tool_calls",
        },
    },
]

def classify_tool_failure(result):
    if result["ok"]:
        return "ok"
    joined = " | ".join(result["errors"])
    if "arguments" in joined:
        return "arguments_parse_error"
    if "name" in joined:
        return "unknown_tool_name"
    return "tool_schema_error"

tool_rows = []
for case in tool_cases:
    result = validate_tool_turn(case["payload"])
    tool_rows.append({
        "case": case["case"],
        "validation_ok": result["ok"],
        "failure_mode": classify_tool_failure(result),
        "tool_count": len(result["normalized"].get("tool_calls", [])),
        "top_error": result["errors"][0] if result["errors"] else "",
    })

tool_results = pd.DataFrame(tool_rows)
tool_results

## Step 9: Adapt normalized tool calls into an OpenAI-compatible response

A response adapter is different from a parser. The parser recovers structure from raw output. The adapter emits a stable provider-facing envelope from your normalized internal state.

In [ ]:
def to_openai_tool_response(turn, model_name="Qwen/Qwen2.5-3B-Instruct"):
    validated = model_validate(NormalizedToolTurn, turn)
    dumped = validated.model_dump() if hasattr(validated, "model_dump") else validated.dict()
    prompt_tokens = 128 + 11 * len(dumped["tool_calls"])
    completion_tokens = 24 + 9 * len(dumped["tool_calls"])
    return {
        "id": "chatcmpl-contract-demo",
        "object": "chat.completion",
        "model": model_name,
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": dumped["content"],
                    "tool_calls": [
                        {
                            "id": call["id"],
                            "type": "function",
                            "function": {
                                "name": call["name"],
                                "arguments": json.dumps(call["arguments"]),
                            },
                        }
                        for call in dumped["tool_calls"]
                    ],
                },
                "finish_reason": dumped["finish_reason"],
            }
        ],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        },
    }

validated_turn = validate_tool_turn(tool_cases[0]["payload"])["normalized"]
openai_preview = to_openai_tool_response(validated_turn)
print(json.dumps(openai_preview, indent=2))

## Step 10: Simulate policy-level contract pass rates

Prompting alone does not guarantee structured outputs. We will compare four open, notebook-friendly policy styles: free-form prompting, JSON prompting, schema-constrained decoding, and a parser-plus-contract gate.

In [ ]:
policy_profiles = pd.DataFrame([
    {"policy": "free_form", "structured_parse_p": 0.74, "structured_schema_p": 0.63, "tool_parse_p": 0.69, "tool_schema_p": 0.61, "repairable_p": 0.30, "base_latency_ms": 620},
    {"policy": "json_prompt_only", "structured_parse_p": 0.86, "structured_schema_p": 0.77, "tool_parse_p": 0.79, "tool_schema_p": 0.72, "repairable_p": 0.36, "base_latency_ms": 655},
    {"policy": "schema_constrained", "structured_parse_p": 0.97, "structured_schema_p": 0.93, "tool_parse_p": 0.91, "tool_schema_p": 0.88, "repairable_p": 0.18, "base_latency_ms": 700},
    {"policy": "parser_plus_contract_gate", "structured_parse_p": 0.99, "structured_schema_p": 0.96, "tool_parse_p": 0.96, "tool_schema_p": 0.94, "repairable_p": 0.10, "base_latency_ms": 735},
])

def simulate_policy_row(policy_row, n_requests=240, seed=0):
    rng = random.Random(seed + int(policy_row["base_latency_ms"]))
    rows = []
    for request_id in range(n_requests):
        task_kind = weighted_pick({"structured": 0.58, "tool": 0.42}, rng)
        parse_key = "structured_parse_p" if task_kind == "structured" else "tool_parse_p"
        schema_key = "structured_schema_p" if task_kind == "structured" else "tool_schema_p"
        parse_ok = rng.random() < policy_row[parse_key]
        schema_ok = parse_ok and rng.random() < policy_row[schema_key]
        contract_ok = parse_ok and schema_ok
        repair_used = False
        if not contract_ok and rng.random() < policy_row["repairable_p"]:
            repair_used = True
        final_ok = contract_ok or repair_used
        latency_ms = policy_row["base_latency_ms"] + rng.uniform(-35, 35) + (42 if repair_used else 0)
        rows.append({
            "policy": policy_row["policy"],
            "request_id": request_id,
            "task_kind": task_kind,
            "contract_ok": contract_ok,
            "repair_used": repair_used,
            "final_ok": final_ok,
            "latency_ms": round(latency_ms, 2),
        })
    return pd.DataFrame(rows)

policy_runs = [simulate_policy_row(row, seed=12) for row in policy_profiles.to_dict("records")]
policy_eval = pd.concat(policy_runs, ignore_index=True)
policy_summary = (
    policy_eval.groupby("policy")
    .agg(
        contract_pass_rate=("contract_ok", "mean"),
        final_success_rate=("final_ok", "mean"),
        repair_rate=("repair_used", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
    )
    .round(3)
    .reset_index()
)
policy_summary

## Step 11: Visualize pass rate versus repair burden

A runtime policy can improve contract pass rate in two different ways: reduce malformed output up front, or repair more aggressively after the fact. Those are not the same operational choice.

In [ ]:
chart = policy_summary.sort_values("contract_pass_rate")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].bar(chart["policy"], chart["contract_pass_rate"], color="#4c78a8", label="contract pass")
axes[0].bar(chart["policy"], chart["final_success_rate"] - chart["contract_pass_rate"], bottom=chart["contract_pass_rate"], color="#f58518", label="repaired")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Contract pass versus repaired success")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend()

axes[1].bar(chart["policy"], chart["repair_rate"], color="#54a24b")
axes[1].set_ylim(0, max(0.05, float(chart["repair_rate"].max()) * 1.25))
axes[1].set_title("Repair burden per policy")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## Step 12: Classify failure modes across both pipelines

Structured JSON failures and tool-call failures are related but not identical. Production dashboards should keep them separate so teams know whether to tune prompts, schemas, parsers, or runtime adapters.

In [ ]:
failure_counts = Counter()
for row in structured_results.to_dict("records"):
    if row["failure_mode"] != "ok":
        failure_counts["structured::" + row["failure_mode"]] += 1
for row in tool_results.to_dict("records"):
    if row["failure_mode"] != "ok":
        failure_counts["tool::" + row["failure_mode"]] += 1

failure_summary = pd.DataFrame(
    [{"failure_mode": name, "count": count} for name, count in failure_counts.items()]
).sort_values(["count", "failure_mode"], ascending=[False, True]).reset_index(drop=True)
failure_summary

## Step 13: Add a conservative repair layer

Repairs should only fix obvious formatting issues. They should not invent missing semantics. We will repair two safe cases: single-string citations and single-quoted JSON tool arguments.

In [ ]:
def attempt_loose_json_load(text):
    try:
        return json.loads(text), None
    except json.JSONDecodeError:
        candidate = text.replace("'", "\"")
        try:
            return json.loads(candidate), None
        except json.JSONDecodeError as exc:
            return None, f"{exc.msg} at char {exc.pos}"

def repair_structured_case(case):
    parsed = parse_json_candidate(case["raw"])
    if parsed["payload"] is None or not isinstance(parsed["payload"], dict):
        return None
    payload = copy.deepcopy(parsed["payload"])
    if isinstance(payload.get("citations"), str):
        payload["citations"] = [payload["citations"]]
    if "needs_human_review" not in payload:
        payload["needs_human_review"] = True
    return payload

def repair_tool_case(payload):
    candidate = copy.deepcopy(payload)
    if "choices" in candidate:
        choice = candidate["choices"][0]
        choice["finish_reason"] = choice.get("finish_reason", "tool_calls")
        raw_calls = choice.get("message", {}).get("tool_calls", [])
    else:
        candidate["finish_reason"] = candidate.get("finish_reason", "tool_calls")
        raw_calls = candidate.get("tool_calls", [])
    for raw_call in raw_calls:
        if "function" in raw_call:
            function_block = raw_call.get("function", {})
            raw_args = function_block.get("arguments")
            if isinstance(raw_args, str):
                repaired_args, error = attempt_loose_json_load(raw_args)
                if error is None:
                    function_block["arguments"] = repaired_args
        else:
            raw_args = raw_call.get("arguments")
            if isinstance(raw_args, str):
                repaired_args, error = attempt_loose_json_load(raw_args)
                if error is None:
                    raw_call["arguments"] = repaired_args
    return candidate

repair_rows = []
for case in structured_cases:
    before = run_structured_pipeline(case)
    repaired_payload = repair_structured_case(case)
    after = validate_structured_payload(repaired_payload) if repaired_payload is not None else {"ok": False}
    repair_rows.append({"case": case["case"], "pipeline": "structured", "before_ok": before["validation_ok"], "after_ok": after["ok"]})
for case in tool_cases:
    before = validate_tool_turn(case["payload"])
    after = validate_tool_turn(repair_tool_case(case["payload"]))
    repair_rows.append({"case": case["case"], "pipeline": "tool", "before_ok": before["ok"], "after_ok": after["ok"]})

repair_summary = pd.DataFrame(repair_rows)
repair_summary

## Step 14: Export a contract report

Later notebooks can reuse these artifacts for deployment gates, routing policy, or incident response.

In [ ]:
export_payload = {
    "structured_schema": structured_schema,
    "structured_results": structured_results.to_dict("records"),
    "tool_results": tool_results.to_dict("records"),
    "policy_summary": policy_summary.to_dict("records"),
    "repair_summary": repair_summary.to_dict("records"),
}
write_json(ARTIFACT_DIR / "contract_report.json", export_payload)
structured_results.to_csv(ARTIFACT_DIR / "structured_results.csv", index=False)
tool_results.to_csv(ARTIFACT_DIR / "tool_results.csv", index=False)
policy_summary.to_csv(ARTIFACT_DIR / "policy_summary.csv", index=False)
print("Saved contract artifacts to", ARTIFACT_DIR.resolve())

## Exercises

1. Add a second structured schema for a multi-item extraction task and compare failure rates.
2. Tighten the repair layer so that missing booleans fail closed instead of receiving defaults.
3. Convert the policy simulation into an SLO gate that blocks deployment when repair rate exceeds a threshold.

In [ ]:
student_checks = pd.DataFrame([
    {"question": "Which layer should reject malformed tool arguments first?", "your_answer": ""},
    {"question": "When is lenient repair acceptable, and when should the request fail closed?", "your_answer": ""},
    {"question": "Which policy gave the highest contract pass rate before repair?", "your_answer": ""},
])
student_checks

## Recap

Structured outputs are not just a prompt trick. They are a systems contract. Parser layers recover candidate structure, validators enforce the schema, adapters emit provider-facing envelopes, and failure reports tell you whether to tune prompts, decoding constraints, or repair policy.

In [ ]:
assert len(structured_results) == len(structured_cases)
assert structured_results["validation_ok"].sum() >= 3
assert policy_summary.sort_values("contract_pass_rate").iloc[-1]["policy"] == "parser_plus_contract_gate"
assert repair_summary["after_ok"].sum() >= repair_summary["before_ok"].sum()
assert tool_results["failure_mode"].isin(["ok", "arguments_parse_error", "unknown_tool_name", "tool_schema_error"]).all()
print("Notebook 12 sanity checks passed.")